# Scala UDF
このノートでは、Scala UDFの例としてバイナリデータのデコーディングを行う。<br>

## Scala とは
Scalaとは、Java仮想マシン(JVM)上で動作し、既存のJava言語ライブラリを利用できる互換性を持った上で<br>
近代的な関数型プログラミングを取り入れるために作られた言語である。<br>
基本的にJavaよりもよりシンプルな記述でプログラミングを行うことができる。

Scala UDFでは、コードを事前にコンパイルしてJAVAアプリケーションファイル(.jar)を生成しておくので、pythonのUDFより実行速度が速い。
## Scala のインストール
ターミナルで`spark-shell`を開くと使われているscalaのバージョンを確かめられる。<br>
これに対応したScalaをConda環境にインストールする。Linuxの場合以下のコマンドで可能。
```
curl -fL https://github.com/coursier/coursier/releases/latest/download/cs-x86_64-pc-linux.gz | gzip -d > cs && chmod +x cs && ./cs install scala:2.13.16 scalac:2.13.16 --install-dir ./temp_bin && ./cs setup --install-dir ./temp_bin && mv ./cs $CONDA_PREFIX/bin/ && mv ./temp_bin/* $CONDA_PREFIX/bin/ && rm -r ./temp_bin
```
"Should we add ... to your PATH ...? [Y/n]"と聞かれるがnでよい。

## Scalaでのコーディング
実際のコーディングは以下のファイルを参照
- scala_build/src/main/scala/decoderUDF.scala
パッケージのビルドに関しては
- scala_build/build.sbt
ファイルにdependency等を記述した上で
```
cd scala_build && sbt package
```
とする。<br>
コンパイルが無事終了すると、
- scala_build/target/scala-2.13/example-scala-udf_2.13-1.0.jar
のように.jarファイルが生成される。<br>

pysparkで利用するには、SparkSessionを作る際にconfig()で"spark.jars"オプションにjarファイルを指定する。


In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .config("spark.jars", "./scala_build/target/scala-2.13/example-scala-udf_2.13-1.0.jar") \
    .getOrCreate()

df = spark.read.parquet("./data/segdata.parquet")
df.show(5)

+--------+---+---+---+---+--------------------+
|event_id| fp|dev|det|mod|             segdata|
+--------+---+---+---+---+--------------------+
|       2| 63|  0|  1| 24|[07 00 00 40 9B 0...|
|       2| 63|  0|  1| 24|[08 00 00 40 34 0...|
|       2| 63|  0| 60|  8|[BC C9 96 AE A6 1...|
|       2| 63| 11| 31| 25|[00 00 00 40 0C 0...|
|       2| 63| 11| 63| 36|[01 00 00 00 01 0...|
+--------+---+---+---+---+--------------------+
only showing top 5 rows


25/10/22 22:49:00 INFO InMemoryFileIndex: It took 0 ms to list leaf files for 1 paths.
25/10/22 22:49:00 INFO SparkContext: Starting job: parquet at NativeMethodAccessorImpl.java:0
25/10/22 22:49:00 INFO DAGScheduler: Got job 2 (parquet at NativeMethodAccessorImpl.java:0) with 1 output partitions
25/10/22 22:49:00 INFO DAGScheduler: Final stage: ResultStage 2 (parquet at NativeMethodAccessorImpl.java:0)
25/10/22 22:49:00 INFO DAGScheduler: Parents of final stage: List()
25/10/22 22:49:00 INFO DAGScheduler: Missing parents: List()
25/10/22 22:49:00 INFO DAGScheduler: Submitting ResultStage 2 (MapPartitionsRDD[7] at parquet at NativeMethodAccessorImpl.java:0), which has no missing parents
25/10/22 22:49:00 INFO MemoryStore: Block broadcast_3 stored as values in memory (estimated size 114.8 KiB, free 433.9 MiB)
25/10/22 22:49:00 INFO MemoryStore: Block broadcast_3_piece0 stored as bytes in memory (estimated size 41.4 KiB, free 433.8 MiB)
25/10/22 22:49:00 INFO SparkContext: Created broadc

このサンプルデータは理研ridfフォーマットのデータの内、segdata部分をバイナリデータとして"segdata"列に書き込んだものである。

In [ ]:
from pyspark.sql import functions as F
# Scalaで定義したUDFを登録する関数を呼ぶ
spark._jvm.decoders.V1190Decoder.registerUDF(spark._jsparkSession)

# "decode_v1190_segdata()"が使えるようになる。
df_decoded = df.filter("mod==24") \
    .select("event_id","fp","dev","det",F.expr("decode_v1190_segdata(segdata)"))
df_decoded.show(10, truncate=False)

25/10/15 17:57:13 WARN SimpleFunctionRegistry: The function decode_v1190_segdata replaced a previously registered function.


+--------+---+---+---+------------------------------------------------------------------------+
|event_id|fp |dev|det|decode_v1190_segdata(segdata)                                           |
+--------+---+---+---+------------------------------------------------------------------------+
|2       |63 |0  |1  |[{7, 0, 6190, 0}, {7, 0, 6736, 1}]                                      |
|2       |63 |0  |1  |[{8, 0, 6046, 0}, {8, 0, 6598, 1}]                                      |
|2       |7  |11 |31 |[{1, 12, 102193, 0}, {1, 12, 103157, 1}]                                |
|2       |8  |11 |31 |[{1, 0, 102262, 0}, {1, 0, 102430, 1}]                                  |
|2       |9  |11 |31 |[{0, 1, 31266, 0}, {0, 1, 33623, 1}, {0, 0, 47360, 0}, {0, 0, 48249, 1}]|
|2       |9  |11 |31 |[{1, 0, 47272, 0}, {1, 0, 48163, 1}]                                    |
|2       |9  |11 |31 |[{2, 0, 47312, 1}, {2, 0, 48214, 0}]                                    |
|2       |9  |11 |31 |[{3, 0, 47298, 0},

explode()を使ってアレイを展開し、ストラクチャーも展開すると以下のようになる。

In [ ]:
df_exploded = df_decoded.select("event_id","fp","dev","det", F.explode("decode_v1190_segdata(segdata)") \
                        .alias("decoded")).select("*","decoded.*").drop("decoded")
df_exploded.show(10,truncate=False)

+--------+---+---+---+---+-------+-----------+----+
|event_id|fp |dev|det|geo|channel|measurement|edge|
+--------+---+---+---+---+-------+-----------+----+
|2       |63 |0  |1  |7  |0      |6190       |0   |
|2       |63 |0  |1  |7  |0      |6736       |1   |
|2       |63 |0  |1  |8  |0      |6046       |0   |
|2       |63 |0  |1  |8  |0      |6598       |1   |
|2       |7  |11 |31 |1  |12     |102193     |0   |
|2       |7  |11 |31 |1  |12     |103157     |1   |
|2       |8  |11 |31 |1  |0      |102262     |0   |
|2       |8  |11 |31 |1  |0      |102430     |1   |
|2       |9  |11 |31 |0  |1      |31266      |0   |
|2       |9  |11 |31 |0  |1      |33623      |1   |
+--------+---+---+---+---+-------+-----------+----+
only showing top 10 rows


以上のように、複雑なUDFはScalaを使って記述することで処理速度の低下を防ぐことができる。